# Singapore HDB Resale Price Analysis and Prediction (2017–2025)

This notebook analyzes Singapore HDB resale transaction data from **2017 to 2025** and builds models to predict resale prices.

## Project goals
- Understand how resale prices changed over time
- Explore the relationship between price, floor area, lease, and estate maturity
- Test whether mature estates have significantly different resale prices
- Build and compare a baseline linear model, an expanded linear model, and a Random Forest model


## 1. Imports and setup


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")


In [2]:
df_hdb = pd.read_csv('hdbprices-2017-2025.csv')
print("Dataset shape:", df_hdb.shape)
df_hdb.head()

Dataset shape: (225608, 11)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


## 2. Data cleaning and feature engineering

Key engineered features:
- `year`: transaction year extracted from `month`
- `remaining_years`: estimated lease remaining based on transaction year
- `is_mature`: whether the town is a mature estate
- `estate_type`: readable label for plotting


In [3]:
df_hdb["month"] = pd.to_datetime(df_hdb["month"])

print("Missing values by column:")
print(df_hdb.isnull().sum())

print("\nDuplicate rows before cleaning:", df_hdb.duplicated().sum())
df_hdb = df_hdb.drop_duplicates().copy()
print("Duplicate rows after cleaning:", df_hdb.duplicated().sum())

Missing values by column:
month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

Duplicate rows before cleaning: 311
Duplicate rows after cleaning: 0


In [4]:
df_hdb["year"] = df_hdb["month"].dt.year

df_hdb["remaining_years"] = 99 - (df_hdb["year"] - df_hdb["lease_commence_date"])

mature_estates = [
    "ANG MO KIO", "BEDOK", "BISHAN", "BUKIT MERAH", "BUKIT TIMAH",
    "CENTRAL AREA", "CLEMENTI", "GEYLANG", "KALLANG/WHAMPOA",
    "MARINE PARADE", "PASIR RIS", "QUEENSTOWN", "SERANGOON",
    "TAMPINES", "TOA PAYOH"
]
df_hdb["is_mature"] = df_hdb["town"].isin(mature_estates).astype(int)
df_hdb["estate_type"] = df_hdb["is_mature"].map({0: "Non-Mature", 1: "Mature"})

# Remove rows with missing values 
model_columns = [
    "resale_price", "month", "year", "lease_commence_date", "remaining_years",
    "floor_area_sqm", "town", "flat_type", "flat_model", "storey_range", "is_mature"
]
df_hdb = df_hdb.dropna(subset=model_columns).copy()

print("Cleaned dataset shape:", df_hdb.shape)
df_hdb.head()


Cleaned dataset shape: (225297, 15)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,year,remaining_years,is_mature,estate_type
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,2017,61,1,Mature
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,2017,60,1,Mature
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,2017,62,1,Mature
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,2017,62,1,Mature
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,2017,62,1,Mature


## 3. Quick dataset overview


In [5]:
df_hdb["resale_price"].describe()


count    2.252970e+05
mean     5.266011e+05
std      1.876748e+05
min      1.400000e+05
25%      3.880000e+05
50%      4.950000e+05
75%      6.300000e+05
max      1.700000e+06
Name: resale_price, dtype: float64

## 4. Exploratory data analysis


In [6]:
px.histogram(
    df_hdb,
    x="resale_price",
    nbins=50,
    title="Distribution of HDB Resale Prices"
)


In [7]:
px.scatter(
    df_hdb.sample(min(len(df_hdb), 15000), random_state=42),
    x="floor_area_sqm",
    y="resale_price",
    opacity=0.35,
    title="Resale Price vs Floor Area (sampled for plotting)"
)


In [8]:
yearly_prices = df_hdb.groupby("year", as_index=False)["resale_price"].mean()

px.line(
    yearly_prices,
    x="year",
    y="resale_price",
    markers=True,
    title="Average HDB Resale Price by Year"
)


In [9]:
px.box(
    df_hdb.sample(min(len(df_hdb), 15000), random_state=42),
    x="estate_type",
    y="resale_price",
    title="Resale Price: Mature vs Non-Mature Estates"
)


## 5. Hypothesis test: mature vs non-mature estates

- **H0:** The mean resale price is the same for mature and non-mature estates
- **H1:** The mean resale price is different


In [10]:
mature_prices = df_hdb[df_hdb["is_mature"] == 1]["resale_price"]
non_mature_prices = df_hdb[df_hdb["is_mature"] == 0]["resale_price"]

t_stat, p_value = stats.ttest_ind(
    mature_prices,
    non_mature_prices
)

mean_diff = mature_prices.mean() - non_mature_prices.mean()

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"Mature estate mean: ${mature_prices.mean():,.0f}")
print(f"Non-mature mean: ${non_mature_prices.mean():,.0f}")
print(f"Mean difference: ${mean_diff:,.0f}")

alpha = 0.05
if p_value < alpha:
    print("\nResult: Reject the null hypothesis.")
    print("There is a statistically significant difference between mature and non-mature estate prices.")
else:
    print("\nResult: Fail to reject the null hypothesis.")
    print("There is no statistically significant difference between the two groups.")


t-statistic: 88.2604
p-value: 0.0000e+00
Mature estate mean: $567,191
Non-mature mean: $497,601
Mean difference: $69,591

Result: Reject the null hypothesis.
There is a statistically significant difference between mature and non-mature estate prices.


## 6. Train-test split

To avoid overly optimistic results, all predictive models below are evaluated on a **held-out test set**.


In [11]:
SEED = 42

train_df, test_df = train_test_split(df_hdb, test_size=0.2, random_state=SEED)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (180237, 15)
Test shape: (45060, 15)


## 7. Baseline model: simple OLS regression

This baseline uses only a few intuitive predictors:
- `remaining_years`
- `floor_area_sqm`
- `is_mature`


In [12]:
baseline_features = ["remaining_years", "floor_area_sqm", "is_mature"]

X_train_base = sm.add_constant(train_df[baseline_features])
X_test_base = sm.add_constant(test_df[baseline_features], has_constant="add")
y_train = train_df["resale_price"]
y_test = test_df["resale_price"]

ols_baseline = sm.OLS(y_train, X_train_base).fit()
print(ols_baseline.summary())


                            OLS Regression Results                            
Dep. Variable:           resale_price   R-squared:                       0.532
Model:                            OLS   Adj. R-squared:                  0.532
Method:                 Least Squares   F-statistic:                 6.821e+04
Date:                Sat, 14 Mar 2026   Prob (F-statistic):               0.00
Time:                        15:42:05   Log-Likelihood:            -2.3761e+06
No. Observations:              180237   AIC:                         4.752e+06
Df Residuals:                  180233   BIC:                         4.752e+06
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            -3.78e+05   2200.889   -1

In [24]:
baseline_pred = ols_baseline.predict(X_test_base)

baseline_results = {
    "Model": "Baseline OLS",
    "MAE": mean_absolute_error(y_test, baseline_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, baseline_pred)),
    "MAPE (%)": np.mean(np.abs((y_test - baseline_pred) / y_test)) * 100,
    "R²": r2_score(y_test, baseline_pred)
}
pd.DataFrame([baseline_results]).round(4)

,Model,MAE,RMSE,MAPE (%),R²
0,Baseline OLS,100136.0534,128120.9821,20.0802,0.5307


## 8. Expanded model: feature-rich OLS regression

This model adds more categorical information:
- `flat_type`
- `flat_model`
- `storey_range`
- `town`

For linear regression, we convert categorical columns into dummy variables.


In [22]:
expanded_train = train_df.copy()
expanded_test = test_df.copy()

categorical_cols = ["flat_type", "flat_model", "storey_range", "town"]
drop_cols = ["resale_price", "month", "block", "street_name", "remaining_lease", "estate_type"]

X_train_exp = pd.get_dummies(
    expanded_train.drop(columns=drop_cols, errors="ignore"),
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)
X_test_exp = pd.get_dummies(
    expanded_test.drop(columns=drop_cols, errors="ignore"),
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

X_train_exp_sm = sm.add_constant(X_train_exp)
X_test_exp_sm = sm.add_constant(X_test_exp, has_constant="add")

ols_expanded = sm.OLS(y_train, X_train_exp_sm).fit()
print(ols_expanded.summary())


                            OLS Regression Results                            
Dep. Variable:           resale_price   R-squared:                       0.867
Model:                            OLS   Adj. R-squared:                  0.867
Method:                 Least Squares   F-statistic:                 1.709e+04
Date:                Sat, 14 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:28:20   Log-Likelihood:            -2.2623e+06
No. Observations:              180237   AIC:                         4.525e+06
Df Residuals:                  180167   BIC:                         4.525e+06
Df Model:                          69                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [23]:
expanded_pred = ols_expanded.predict(X_test_exp_sm)

expanded_results = {
    "Model": "Expanded OLS",
    "MAE": mean_absolute_error(y_test, expanded_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, expanded_pred)),
    "MAPE (%)": np.mean(np.abs((y_test - expanded_pred) / y_test)) * 100,
    "R²": r2_score(y_test, expanded_pred)
}
pd.DataFrame([expanded_results]).round(4)


,Model,MAE,RMSE,MAPE (%),R²
0,Expanded OLS,52294.4025,68286.1664,10.9008,0.8667


## 9. Random Forest model

Random Forest can capture nonlinear relationships and interactions that linear regression may miss.


In [28]:
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train_exp, y_train)
rf_pred = rf.predict(X_test_exp)

rf_results = {
    "Model": "Random Forest",
    "MAE": mean_absolute_error(y_test, rf_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, rf_pred)),
    "MAPE (%)": np.mean(np.abs((y_test - rf_pred) / y_test)) * 100,
    "R²": r2_score(y_test, rf_pred)
}
pd.DataFrame([rf_results]).round(4)


,Model,MAE,RMSE,MAPE (%),R²
0,Random Forest,26761.0528,38451.7147,5.1472,0.9577


## 10. Model comparison


In [29]:
results_df = pd.DataFrame([baseline_results, expanded_results, rf_results])
results_df = results_df.sort_values("MAPE (%)").reset_index(drop=True)
results_df.round(4)


,Model,MAE,RMSE,MAPE (%),R²
0,Random Forest,26761.0528,38451.7147,5.1472,0.9577
1,Expanded OLS,52294.4025,68286.1664,10.9008,0.8667
2,Baseline OLS,100136.0534,128120.9821,20.0802,0.5307


## 11. Feature importance from Random Forest


In [30]:
feature_importance = (
    pd.DataFrame({
        "feature": X_train_exp.columns,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance.head(10)


,feature,importance
0,floor_area_sqm,0.424237
1,year,0.183130
2,is_mature,0.136697
3,lease_commence_date,0.121843
4,flat_model_Model A,0.013491
5,town_TAMPINES,0.011593
6,remaining_years,0.011567
7,town_PASIR RIS,0.008099
8,town_BEDOK,0.007915
9,town_BISHAN,0.006000


In [32]:
px.bar(
    feature_importance.head(10),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 10 Random Forest Feature Importances"
)


## 12. Key findings

- Resale prices show an upward trend over time
- Mature estates have higher average resale prices than non-mature estates
- The expanded OLS model improves materially over the baseline OLS model
- Random Forest achieves the strongest predictive performance among the tested models


## 13. Limitations

- Important features such as exact floor level and distance to MRT are not included
- Results depend on the quality and coverage of the original dataset
- Random Forest is more accurate here, but less interpretable than simple linear regression
